# KWISMO — Modèle B : Entraînement NLP & Auto-Catégorisation

Ce notebook permet d'exécuter le pipeline complet du **Modèle B** :
1. **Nettoyage des données** (`src.data.clean`) sur le jeu de données `data/raw/kwismo_data/messages.jsonl`.
2. **Augmentation synthétique** (`src.data.augment`) en Franglais/Pidgin local.
3. **Entraînement & Sauvegarde** (`src.models.model_b.train`) du modèle NLP et du modèle de repli TF-IDF (priorité à `model_b_augmented.jsonl`).
4. **Métriques d'Évaluation & Graphique de Matrice de Confusion**.
5. **Inférence & Détection Autonome sur des Exemples Bruts Sans Catégories** (avec extraction NER).

In [ ]:
# Détection de l'environnement d'exécution
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if ON_COLAB:
    ENV_NAME = "Google Colab"
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive
            print("⚡ Connexion automatique à Google Drive...")
            drive.mount('/content/drive')
        except Exception as err:
            print(f"⚠️ Montage Google Drive recommandé : {err}")
elif ON_KAGGLE:
    ENV_NAME = "Kaggle Notebooks"
else:
    ENV_NAME = "Local"

print(f"Environnement de calcul détecté : {ENV_NAME}")

In [ ]:
# Configuration du dossier de travail sur Cloud (Colab / Kaggle) et Local
if ON_COLAB:
    PROJECT_DIR = Path("/content/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /content/kwismo
    else:
        !git -C /content/kwismo fetch && git -C /content/kwismo reset --hard origin/main
elif ON_KAGGLE:
    PROJECT_DIR = Path("/kaggle/working/kwismo/kwismo-ai")
    if not PROJECT_DIR.exists():
        !git clone https://github.com/newtonachonduh46/kwismo.git /kaggle/working/kwismo
    else:
        !git -C /kaggle/working/kwismo fetch && git -C /kaggle/working/kwismo reset --hard origin/main
else:
    current = Path.cwd()
    PROJECT_DIR = current
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            PROJECT_DIR = candidate
            break
        elif (candidate / "kwismo-ai" / "src").exists():
            PROJECT_DIR = candidate / "kwismo-ai"
            break

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
print("Dossier racine du projet kwismo-ai :", PROJECT_DIR)

In [ ]:
# Sélection de l'environnement virtuel et injection dynamique des dépendances .venv dans sys.path
VENV_DIR = PROJECT_DIR / ".venv313"

if ON_COLAB or ON_KAGGLE:
    python313_bin = VENV_DIR / "bin" / "python"
    if not python313_bin.exists():
        print("Installation de Python 3.13 et création de l'environnement .venv313...")
        !apt-get update -y
        !apt-get install -y software-properties-common
        !add-apt-repository -y ppa:deadsnakes/ppa
        !apt-get update -y
        !apt-get install -y python3.13 python3.13-venv python3.13-dev
        !python3.13 -m venv {VENV_DIR}
        !{VENV_DIR}/bin/pip install --upgrade pip
        !{VENV_DIR}/bin/pip install -r requirements.txt
    PYTHON_BIN = str(python313_bin)
    site_packages = VENV_DIR / "lib" / f"python3.13" / "site-packages"
    if site_packages.exists() and str(site_packages) not in sys.path:
        sys.path.insert(0, str(site_packages))
else:
    local_venv_win = PROJECT_DIR / ".venv" / "Scripts" / "python.exe"
    local_venv_nix = PROJECT_DIR / ".venv" / "bin" / "python"
    
    if local_venv_win.exists():
        PYTHON_BIN = str(local_venv_win)
        site_pkgs = PROJECT_DIR / ".venv" / "Lib" / "site-packages"
        if site_pkgs.exists() and str(site_pkgs) not in sys.path:
            sys.path.insert(0, str(site_pkgs))
    elif local_venv_nix.exists():
        PYTHON_BIN = str(local_venv_nix)
        site_pkgs = PROJECT_DIR / ".venv" / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
        if site_pkgs.exists() and str(site_pkgs) not in sys.path:
            sys.path.insert(0, str(site_pkgs))
    else:
        PYTHON_BIN = sys.executable

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def run_module(module: str) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run([PYTHON_BIN, "-m", module], cwd=PROJECT_DIR, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"Standard Output:\n{result.stdout}")
        print(f"Standard Error:\n{result.stderr}")
        raise RuntimeError(f"{module} a échoué (code {result.returncode})")
    else:
        print(result.stdout)

ver_proc = subprocess.run([PYTHON_BIN, "--version"], capture_output=True, text=True)
print("Interprète Python configuré :", PYTHON_BIN)
print("Version vérifiée :", ver_proc.stdout.strip() or ver_proc.stderr.strip())
print("Accès sys.path configuré pour le projet et .venv : OK")

## 1. Nettoyage du Jeu de Données

Exécution de `src.data.clean` pour filtrer le bruit web et structurer le jeu de données propre dans `data/processed/model_b_clean.jsonl`.

In [ ]:
run_module("src.data.clean")

## 2. Augmentation du dataset en Franglais/Pidgin

Exécution de `src.data.augment` pour générer des phrases et rendre les données plus pertinentes dans `data/processed/model_b_augmented.jsonl`.

In [ ]:
run_module("src.data.augment")

## 3. Entraînement du Modèle B

Exécution de `src.models.model_b.train` pour entraîner le modèle et mettre à jour le registre `models/registry.json`. (Sélectionne prioritairement `model_b_augmented.jsonl`).

In [ ]:
run_module("src.models.model_b.train")

## 4. Métriques d'Évaluation & Graphes de Performance

Calcul des métriques globales (Précision, Rappel, F1-score) et affichage du graphique de la **Matrice de Confusion** du Modèle B sur le jeu de données d'entraînement.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from src.models.model_b.train import load_cleaned_dataset
from src.models.model_b.preprocess import categorize_description

# Chargement des données et calcul des prédictions
texts, true_categories, _ = load_cleaned_dataset()
pred_categories = [categorize_description(t) for t in texts]

print(f"=== Rapport de Classification complet (sur {len(texts)} exemples) ===")
print(classification_report(true_categories, pred_categories))

# Affichage du graphique de la matrice de confusion
labels = sorted(list(set(true_categories + pred_categories)))
cm = confusion_matrix(true_categories, pred_categories, labels=labels)

fig, ax = plt.subplots(figsize=(10, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap="Greens", ax=ax, xticks_rotation=45)
plt.title("Matrice de Confusion — Modèle B (Auto-Catégorisation)")
plt.tight_layout()
plt.show()

## 5. Inférence Autonome sur des Exemples Bruts Sans Catégorie Préalable

Évaluation du Modèle B sur une série de messages bruts sans étiquette : l'IA détermine en toute autonomie s'il s'agit d'une arnaque, lui attribue sa catégorie et extrait les entités (montants, USSD, numéros cibles).

In [ ]:
from src.models.model_b.preprocess import categorize_description, extract_entities

# Exemples bruts SANS catégorie transmise à l'IA
raw_unlabeled_examples = [
    "Mobile Money, vous avez reçu 75 000 FCFA de Paul. Nouveau solde 80 000 F. Tapez *126# pour vérifier.",
    "Bonjour je suis agent Orange Money, votre compte présente un problème. Envoyez votre code secret PIN au 694000000.",
    "Prime Promo MTN : numéro gagnant sélectionné pour recevoir la somme de 400 000 F CFA.",
    "Mon numéro a été dupliqué par une attaque SIM swap et ma carte SIM est bloquée chez l'opérateur.",
    "Offre d'emploi fictive demandant 25 000 FCFA de frais de dossier pour un entretien à Douala.",
    "Sensibilisation MTN MoMo : numéros officiels d'assistance et règles de sécurité. Aucun agent ne demande votre PIN.",
    "Faux profil Facebook usurpe mon identité et demande de l'argent à mes proches.",
]

print("=== Détection et Catégorisation Autonome par l'IA ===\n")
results_data = []

for idx, raw_text in enumerate(raw_unlabeled_examples, start=1):
    detected_cat = categorize_description(raw_text)
    entities = extract_entities(raw_text)
    is_fraud = detected_cat != "legitimate_info" and detected_cat != "unknown_scam_pattern"
    
    results_data.append({
        "Exemple": f"Ex #{idx}",
        "Est une Arnaque ?": "🔴 OUI (Fraude)" if is_fraud else "🟢 NON (Sensibilisation)",
        "Catégorie Détectée": detected_cat,
        "Montants Extraits": entities["montants"],
        "Codes USSD": entities["codes_ussd"],
        "Numéros Cibles": entities["numeros_cibles"],
        "Texte Brut": raw_text
    })

df_results = pd.DataFrame(results_data)
display(df_results[["Exemple", "Est une Arnaque ?", "Catégorie Détectée", "Montants Extraits", "Codes USSD", "Numéros Cibles"]])